# Pokemon TCG AI Battle — Agent Testing

Run this notebook inside a Kaggle competition notebook. The cabt engine is pre-installed there.

**Steps:**
1. Set `GITHUB_URL` to your repository.
2. Edit `AGENTS` to list the agent class names you want to test.
3. Run all cells.

## 1. Clone repo

In [ ]:
GITHUB_URL = "https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git"  # replace this

!git clone {GITHUB_URL} /kaggle/working/repo

In [ ]:
import os
import sys

os.chdir("/kaggle/working/repo")
sys.path.insert(0, "/kaggle/working/repo/src")

## 2. Available deck

Card IDs in the engine's bundled default deck.

In [ ]:
from collections import Counter
from kaggle_environments.envs.cabt.cabt import deck as DEFAULT_DECK

counts = Counter(DEFAULT_DECK)
print(f"Default deck — {len(DEFAULT_DECK)} cards total\n")
print(f"{'Card ID':<10} {'Count'}")
print("-" * 20)
for card_id, count in sorted(counts.items()):
    print(f"{card_id:<10} {count}")

## 3. Configure agents

Add the class names of the agents you want to test. Each name must match a class
in `src/ptcg/agents/` using the snake_case naming convention:
`RandomAgent` → `src/ptcg/agents/random_agent.py`.

In [ ]:
AGENTS = [
    "RandomAgent",
]

## 4. Run battles

Every agent in the list plays against every other agent (including itself).

In [ ]:
import importlib
import re
from kaggle_environments import make


def load_agent(class_name: str):
    module_name = re.sub(r"(?<!^)(?=[A-Z])", "_", class_name).lower()
    module = importlib.import_module(f"ptcg.agents.{module_name}")
    return getattr(module, class_name)()


def run_battle(agent0, agent1):
    env = make("cabt", configuration={"decks": [agent0.get_deck(), agent1.get_deck()]})
    env.run([agent0, agent1])
    rewards = [step["reward"] for step in env.steps[-1]]
    return rewards, env


results = []

for name0 in AGENTS:
    for name1 in AGENTS:
        agent0 = load_agent(name0)
        agent1 = load_agent(name1)

        agent0.on_game_start()
        agent1.on_game_start()

        rewards, env = run_battle(agent0, agent1)
        result = {"steps": env.steps, "rewards": rewards}

        agent0.on_game_end(result)
        agent1.on_game_end(result)

        results.append({"agent0": name0, "agent1": name1, "rewards": rewards})
        print(f"{name0} vs {name1}  →  {name0}: {rewards[0]}  |  {name1}: {rewards[1]}")

## 5. Results summary

In [ ]:
print(f"{'Agent 0':<20} {'Agent 1':<20} {'Reward 0':>10} {'Reward 1':>10}")
print("-" * 65)
for r in results:
    print(f"{r['agent0']:<20} {r['agent1']:<20} {r['rewards'][0]:>10} {r['rewards'][1]:>10}")